# AWS Glue Notebook — Camada Silver: State of Data Brasil (2023 · 2024 · 2025)

Este notebook consulta as tabelas **`tb_pesquisa_2023_bronze`**, **`tb_pesquisa_2024_bronze`** e **`tb_pesquisa_2025_bronze`** já catalogadas no **AWS Glue Data Catalog**, padroniza o schema das três edições da pesquisa (seleção e renomeação de colunas), harmoniza valores categóricos, **une as bases (union, não join)** e grava uma nova tabela em formato **Parquet** no Amazon S3, deixando-a catalogada para consulta no Athena.

Além do perfil demográfico/profissional, a tabela traz os blocos de **tecnologias (linguagens)** e **uso de IA Generativa** (empresa, indivíduo e produtividade pessoal), resolvidos a partir dos nomes de coluna da Bronze pelo *sufixo descritivo* da pergunta — que é estável entre edições mesmo quando o código da pergunta muda de posição.

**Resultado esperado:** tabela `workspace.tb_state_of_data_silver` gravada no S3 em `s3://614133392546-lab/data-output/silver/state-of-data/`, particionada por `ano_pesquisa`, com 14.005 linhas (5.293 + 5.217 + 3.495).

## 1. Inicialização da sessão Glue / Spark

Execute esta célula para iniciar a sessão interativa do AWS Glue.

In [1]:
%idle_timeout 15
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5
%region us-east-1
%iam_role arn:aws:iam::614133392546:role/LabRole

import sys
import re
import json
import unicodedata
import boto3
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType, BooleanType

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel


For more information on available magic commands, please type %help in any new cell.



Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html


Current idle_timeout is None minutes.


idle_timeout has been set to 15 minutes.


...

Applying the following default arguments:


--glue_kernel_version 1.0.9


--enable-glue-datacatalog true


Waiting for session 37d92bd3-90fb-43e8-9e78-fa4e10568929 to get into ready status...


Session 37d92bd3-90fb-43e8-9e78-fa4e10568929 has been created.


## 2. Parâmetros do processamento

Centralize aqui os nomes de banco, tabelas de origem e destino. Caso necessário, altere apenas esta célula.

In [7]:
# bucket <account_id>-lab (mesma convenção da bronze)
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
BUCKET_NAME = f"{ACCOUNT_ID}-lab"

DATABASE_NAME = "workspace"

TABLES_BRONZE = {
    2023: "tb_pesquisa_2023_bronze",
    2024: "tb_pesquisa_2024_bronze",
    2025: "tb_pesquisa_2025_bronze",
}

OUTPUT_TABLE = "tb_state_of_data_silver"
OUTPUT_FULL_TABLE_NAME = f"{DATABASE_NAME}.{OUTPUT_TABLE}"

OUTPUT_PATH = f"s3://{BUCKET_NAME}/data-output/silver/state-of-data/"

# json com o mapeamento bronze->silver que foi de fato usado (pra rastreabilidade)
METADATA_KEY = "data-output/silver/_metadata/tb_state_of_data_silver_colunas.json"

print(f"Tabelas de origem: {list(TABLES_BRONZE.values())}")
print(f"Tabela de destino: {OUTPUT_FULL_TABLE_NAME}")
print(f"Caminho S3: {OUTPUT_PATH}")
print(f"Metadados: s3://{BUCKET_NAME}/{METADATA_KEY}")

Tabelas de origem: ['tb_pesquisa_2023_bronze', 'tb_pesquisa_2024_bronze', 'tb_pesquisa_2025_bronze']
Tabela de destino: workspace.tb_state_of_data_silver
Caminho S3: s3://614133392546-lab/data-output/silver/state-of-data/
Metadados: s3://614133392546-lab/data-output/silver/_metadata/tb_state_of_data_silver_colunas.json


## 3. Mapeamento de colunas Bronze → Silver e regras de harmonização (perfil)

- **Mapeamento**: nome exato da coluna de origem em cada ano (conforme `data-output/bronze/_metadata/pesquisa-<ano>_colunas.json`). `None` indica que a pergunta não existe naquela edição (a coluna é criada com `lit(None)`).
- **Typos de faixa salarial**: dois rótulos com 1 ocorrência cada, corrigidos antes de qualquer outro tratamento.
- **Tabela de referência de faixa salarial**: ordem (Int) e ponto médio (Double) para permitir ordenação e cálculo de médias no Athena.

> A resolução dos nomes de origem é **case-insensitive**: o Glue Catalog armazena os nomes de coluna em minúsculas (padrão do Hive metastore), enquanto os arquivos Parquet preservam o case original. Assim o mapeamento funciona tanto lendo via `from_catalog` quanto via `spark.table`.

In [7]:
# coluna silver -> nome exato na bronze de cada ano (tirado dos json _metadata da bronze)
COLUMN_MAPPING = {
    "id_resposta":              {2023: "P0_id",                                                      2024: "0_a_token",                                2025: "0_a_token"},
    "genero":                   {2023: "P1_b_Genero",                                                2024: "1_b_genero",                               2025: "1_b_genero"},
    "cor_raca_etnia":           {2023: "P1_c_Cor_raca_etnia",                                        2024: "1_c_cor_raca_etnia",                       2025: "1_c_cor_raca_etnia"},
    "uf_onde_mora":             {2023: "P1_i_1_uf_onde_mora",                                        2024: "1_i_1_uf_onde_mora",                       2025: "1_i_1_uf_onde_mora"},
    "regiao_onde_mora":         {2023: "P1_i_2_Regiao_onde_mora",                                    2024: "1_i_2_regiao_onde_mora",                   2025: "1_i_2_regiao_onde_mora"},
    "nivel_ensino":             {2023: "P1_l_Nivel_de_Ensino",                                       2024: "1_l_nivel_de_ensino",                      2025: "1_l_nivel_de_ensino"},
    "area_formacao":            {2023: "P1_m_Área_de_Formação",                                      2024: "1_m_área_de_formação",                     2025: "1_m_área_de_formação"},
    "cargo_atual":              {2023: "P2_f_Cargo_Atual",                                           2024: "2_f_cargo_atual",                          2025: "2_f_cargo_atual"},
    "nivel_senioridade":        {2023: "P2_g_Nivel",                                                 2024: "2_g_nivel",                                2025: "2_g_nivel"},
    "faixa_salarial":           {2023: "P2_h_Faixa_salarial",                                        2024: "2_h_faixa_salarial",                       2025: "2_h_faixa_salarial"},
    "setor":                    {2023: "P2_b_Setor",                                                 2024: "2_b_setor",                                2025: "2_b_setor"},
    "numero_funcionarios":      {2023: "P2_c_Numero_de_Funcionarios",                                2024: "2_c_numero_de_funcionarios",               2025: "2_c_numero_de_funcionarios"},
    "tempo_experiencia_dados":  {2023: "P2_i_Quanto_tempo_de_experiência_na_área_de_dados_você_tem", 2024: "2_i_tempo_de_experiencia_em_dados",        2025: "2_i_tempo_de_experiencia_em_dados"},
    "modelo_trabalho_atual":    {2023: None,                                                         2024: "2_r_modelo_de_trabalho_atual",             2025: "2_q_modelo_de_trabalho_atual"},
    "ia_generativa_prioridade": {2023: "P3_e_AI_Generativa_é_uma_prioridade_em_sua_empresa",         2024: "3_e_ai_generativa_e_llm_é_uma_prioridade", 2025: "3_e_ai_generativa_e_llm_é_uma_prioridade"},
}

PARTITION_COLUMN = "ano_pesquisa"

# Regra 4: em 2023 'Engenheiro de Dados' e 'Arquiteto de Dados' eram uma opção só; de 2024 em diante
# viraram duas. Sem juntar, o eng de dados 'some' em 2024 no gráfico. Os outros cargos ficam iguais (NULL inclusive).
CARGO_ENGENHEIRO_ARQUITETO = "Engenheiro de Dados/Arquiteto de Dados"
CARGOS_ENGENHEIRO_ARQUITETO_VARIANTES = [
    "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",   # 2023
    "Engenheiro de Dados/Data Engineer/Data Architect",                      # 2024/2025
    "Arquiteto de Dados/Data Architect",                                     # 2024/2025
]

# Regra 2: dois typos que vieram do kaggle (1 registro cada) - achei rodando DISTINCT na bronze
TYPOS_FAIXA_SALARIAL = {
    "de R$ 101/mês a R$ 2.000/mês":    "de R$ 1.001/mês a R$ 2.000/mês",
    "de R$ 25.001/mês a R$ 3000/mês":  "de R$ 25.001/mês a R$ 30.000/mês",
}

# Regra 3: ordem e ponto médio de cada faixa, pra dar pra ordenar e tirar média no athena
FAIXA_SALARIAL_REF = [
    ("Menos de R$ 1.000/mês",             1,   500.0),
    ("de R$ 1.001/mês a R$ 2.000/mês",    2,  1500.5),
    ("de R$ 2.001/mês a R$ 3.000/mês",    3,  2500.5),
    ("de R$ 3.001/mês a R$ 4.000/mês",    4,  3500.5),
    ("de R$ 4.001/mês a R$ 6.000/mês",    5,  5000.5),
    ("de R$ 6.001/mês a R$ 8.000/mês",    6,  7000.5),
    ("de R$ 8.001/mês a R$ 12.000/mês",   7, 10000.5),
    ("de R$ 12.001/mês a R$ 16.000/mês",  8, 14000.5),
    ("de R$ 16.001/mês a R$ 20.000/mês",  9, 18000.5),
    ("de R$ 20.001/mês a R$ 25.000/mês", 10, 22500.5),
    ("de R$ 25.001/mês a R$ 30.000/mês", 11, 27500.5),
    ("de R$ 30.001/mês a R$ 40.000/mês", 12, 35000.5),
    ("Acima de R$ 40.001/mês",           13, 45000.0),
]

# ordem das colunas de perfil na silver
PERFIL_COLUMNS = [
    "id_resposta",
    "genero",
    "cor_raca_etnia",
    "uf_onde_mora",
    "regiao_onde_mora",
    "nivel_ensino",
    "area_formacao",
    "cargo_atual",
    "cargo_atual_agrupado",
    "nivel_senioridade",
    "nivel_senioridade_agrupada",
    "faixa_salarial",
    "faixa_salarial_ordem",
    "faixa_salarial_ponto_medio",
    "setor",
    "numero_funcionarios",
    "tempo_experiencia_dados",
    "modelo_trabalho_atual",
    "ia_generativa_prioridade",
]

print(f"{len(COLUMN_MAPPING)} colunas de perfil mapeadas + partição '{PARTITION_COLUMN}'")
print(f"{len(TYPOS_FAIXA_SALARIAL)} typos de faixa salarial | {len(FAIXA_SALARIAL_REF)} faixas de referência")

15 colunas de perfil mapeadas + partição 'ano_pesquisa'
2 typos de faixa salarial | 13 faixas de referência


## 4. Extensão: tecnologias e IA Generativa — resolução de colunas por sufixo descritivo

Os sub-itens da Bronze seguem o padrão **`<código_pergunta>_<texto_descritivo>`** (ex.: `P3_f_1_Colaboradores_...` em 2023, `3_f_1_Colaboradores_...` em 2024, `4_i_3_Desenvolvedores_...` em 2025). O texto descritivo é idêntico entre edições mesmo quando o código muda de posição (`3_g` → `3_h` em 2025). Estratégia:

1. Para cada coluna da Bronze, separa-se o código da pergunta do sufixo descritivo com a regex `^[A-Za-z]?\d+(_[A-Za-z]{1,2})?(_\d+)*_` e normaliza-se o sufixo (minúsculas, sem acento).
2. Cada campo alvo declara **seção** (3 = empresa, 4 = indivíduo/tecnologia), uma **regra** sobre o sufixo (regex) e, para sub-itens, uma **coluna-âncora** cuja pergunta deve ser a mesma — evita, por exemplo, que `tec_usa_nenhuma` resolva para "Não utilizo nenhuma ferramenta de BI".
3. Campos com regra `nome` são resolvidos pelo nome completo (exceções manuais). Se nada casa, a coluna vira `lit(None)` naquele ano.
4. Sub-itens (`"1"`/`"0"` como string) são convertidos para **boolean**; `NULL` permanece `NULL`.
5. Cada campo declara os **anos aplicáveis**; fora deles a coluna é `lit(None)` sem tentar resolver.

> **Nota metodológica — tecnologia.** A partir de 2025, a pesquisa não repete a pergunta "linguagens usadas no dia a dia" (substituída por "linguagem preferida", de natureza diferente: os sub-itens `4_c_*` pertencem a `4_c_linguagem_preferida`). Por isso `tec_usa_*` só existe em 2023–2024 e `tec_pref_*` só em 2025; a comparação de adoção de tecnologia 2023→2025 fica limitada a 2023–2024, e 2025 é reportado separadamente como "linguagem preferida". Em 2024 o sub-item "Não utilizo nenhuma…" não foi codificado na fonte (100% `"0"`), então `tec_usa_nenhuma` é derivado da guarda-chuva `4_d_linguagem_de_programacao_dia_a_dia`. Para 2023/2024, `tec_pref_normalizada` comprime o texto livre de "linguagem preferida" nos valores mais frequentes.

Ao final, `ia_pessoal_nivel_uso` é derivada dos 5 booleanos de produtividade pessoal.

In [7]:
# Os nomes na bronze seguem "<codigo da pergunta>_<texto>" (P3_f_1_Colaboradores... / 3_f_1_... / 4_i_3_...).
# O texto é o mesmo entre os anos mesmo quando o código muda de posição (3_g virou 3_h em 2025),
# então resolvo as colunas pelo texto e não pelo código.
# grupo 1 = código, grupo 2 = seção (3 = empresa, 4 = indivíduo), grupo 3 = sufixo
# esse regex ficou meio feio mas resolveu: a letra da pergunta é opcional e o índice do sub-item pode não existir
CODIGO_PERGUNTA_RE = re.compile(r"^([A-Za-z]?(\d+)(?:_[A-Za-z]{1,2})?(?:_\d+)*)_(.+)$")


def normalizar_texto(txt: str) -> str:
    # minúsculas e sem acento: 'Não_sei_opinar' -> 'nao_sei_opinar'
    return unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode().lower()


def codigo_pergunta_base(codigo: str) -> str:
    # 'P3_f_1' -> 'P3_f' (a pergunta sem o índice do sub-item)
    return re.sub(r"(_\d+)+$", "", codigo)


def indexar_sufixos(df):
    indice = []
    for col in df.columns:
        m = CODIGO_PERGUNTA_RE.match(col)
        if m:
            indice.append({"col": col, "codigo": m.group(1), "secao": m.group(2), "sufixo": normalizar_texto(m.group(3))})
    return indice


# (nome_silver, tipo, seção, regra, âncora, anos)
#   regra "sufixo" = regex no sufixo normalizado; "nome" = lista de nomes completos
#   âncora = campo já resolvido que tem que estar na MESMA pergunta (senão tec_usa_nenhuma pegava
#   o "Não utilizo nenhuma ferramenta de BI", que também está na seção 4)
#   anos = None é todos; fora deles vira lit(None) direto, sem nem tentar resolver
B, S = "boolean", "string"
USA = (2023, 2024)   # "linguagens usadas no dia a dia" só existe em 2023/2024
PREF = (2025,)       # sub-itens 4_c_* de 2025 pertencem a "linguagem preferida"
CAMPOS_EXTENSAO = [
    # tecnologia - uso no dia a dia. Só 2023/2024: em 2025 a pergunta não existe mais, os sub-itens 4_c_*
    # de 2025 são da pergunta de linguagem PREFERIDA (descobri cruzando com a guarda-chuva no athena)
    ("tec_usa_python",           B, "4", ("sufixo", r"^python$"),                      None,             USA),
    ("tec_usa_sql",              B, "4", ("sufixo", r"^sql$"),                         "tec_usa_python", USA),
    ("tec_usa_r",                B, "4", ("sufixo", r"^r$"),                           "tec_usa_python", USA),
    ("tec_usa_c_cpp_csharp",     B, "4", ("sufixo", r"^c_c_c$"),                       "tec_usa_python", USA),
    ("tec_usa_julia",            B, "4", ("sufixo", r"^julia$"),                       "tec_usa_python", USA),
    ("tec_usa_vba",              B, "4", ("sufixo", r"^visual_basic_vba$"),            "tec_usa_python", USA),
    ("tec_usa_scala",            B, "4", ("sufixo", r"^scala$"),                       "tec_usa_python", USA),
    ("tec_usa_rust",             B, "4", ("sufixo", r"^rust$"),                        "tec_usa_python", USA),
    ("tec_usa_nenhuma",          B, "4", ("sufixo", r"^nao_utilizo.*linguag"),         "tec_usa_python", USA),  # 2024: sobrescrito pela guarda-chuva (ver TEC_USA_NENHUMA_GUARDA_CHUVA)
    # linguagens que saíram da lista em 2025
    ("tec_usa_dotnet",           B, "4", ("sufixo", r"^net$"),                         "tec_usa_python", USA),
    ("tec_usa_java",             B, "4", ("sufixo", r"^java$"),                        "tec_usa_python", USA),
    ("tec_usa_sas_stata",        B, "4", ("sufixo", r"^sas_stata$"),                   "tec_usa_python", USA),
    ("tec_usa_matlab",           B, "4", ("sufixo", r"^matlab$"),                      "tec_usa_python", USA),
    ("tec_usa_php",              B, "4", ("sufixo", r"^php$"),                         "tec_usa_python", USA),
    ("tec_usa_javascript",       B, "4", ("sufixo", r"^javascript$"),                  "tec_usa_python", USA),
    # categórica, resolvida pelo nome completo mesmo
    ("tec_linguagem_mais_usada", S, "4", ("nome", ["P4_e_Entre_as_linguagens_listadas_abaixo_qual_é_a_que_você_mais_utiliza_no_trabalho",
                                                   "4_e_linguagem_mais_usada"]),        None,             USA),
    # tecnologia - linguagem preferida (multi-select), só 2025
    ("tec_pref_python",          B, "4", ("sufixo", r"^python$"),                      None,              PREF),
    ("tec_pref_sql",             B, "4", ("sufixo", r"^sql$"),                         "tec_pref_python", PREF),
    ("tec_pref_r",               B, "4", ("sufixo", r"^r$"),                           "tec_pref_python", PREF),
    ("tec_pref_c_cpp_csharp",    B, "4", ("sufixo", r"^c_c_c$"),                       "tec_pref_python", PREF),
    ("tec_pref_julia",           B, "4", ("sufixo", r"^julia$"),                       "tec_pref_python", PREF),
    ("tec_pref_vba",             B, "4", ("sufixo", r"^visual_basic_vba$"),            "tec_pref_python", PREF),
    ("tec_pref_scala",           B, "4", ("sufixo", r"^scala$"),                       "tec_pref_python", PREF),
    ("tec_pref_dax",             B, "4", ("sufixo", r"^dax$"),                         "tec_pref_python", PREF),
    ("tec_pref_rust",            B, "4", ("sufixo", r"^rust$"),                        "tec_pref_python", PREF),
    ("tec_pref_nenhuma",         B, "4", ("sufixo", r"^nao_utilizo.*linguag"),         "tec_pref_python", PREF),
    # texto livre da preferida em 2023/2024 - coluna auxiliar, vira tec_pref_normalizada e não vai pra silver
    ("_tec_pref_texto_livre",    S, "4", ("nome", ["P4_f_Entre_as_linguagens_listadas_abaixo_qual_é_a_sua_preferida",
                                                   "4_f_linguagem_preferida"]),         None,             USA),
    # IA na empresa (seção 3)
    ("ia_emp_uso_independente",       B, "3", ("sufixo", r"usando_ai_generativa_de_forma_independente_e_descentralizada"), None, None),
    ("ia_emp_direcionamento_central", B, "3", ("sufixo", r"direcionamento_centralizado_do_uso_de_ai_generativa"), "ia_emp_uso_independente", None),
    ("ia_emp_copilots_dev",           B, "3", ("sufixo", r"desenvolvedores_utilizando_copilots"),                 "ia_emp_uso_independente", None),
    ("ia_emp_produto_externo",        B, "3", ("sufixo", r"melhorar_produtos_externos"),                          "ia_emp_uso_independente", None),
    ("ia_emp_produto_interno",        B, "3", ("sufixo", r"melhorar_produtos_internos_para_os_colaboradores"),    "ia_emp_uso_independente", None),
    ("ia_emp_principal_frente",       B, "3", ("sufixo", r"como_principal_frente_do_negocio"),                    "ia_emp_uso_independente", None),
    ("ia_emp_nao_prioridade",         B, "3", ("sufixo", r"nao_e_prioridade"),                                    "ia_emp_uso_independente", None),
    ("ia_emp_nao_sei_opinar",         B, "3", ("sufixo", r"nao_sei_opinar_sobre_o_uso_de_ia_generativa_e_llms_na_empresa"), "ia_emp_uso_independente", None),
    # motivos pra não usar IA (3_g em 2023/2024, 3_h em 2025)
    ("ia_emp_motivo_casos_uso",              B, "3", ("sufixo", r"^falta_de_compreensao_dos_casos_de_uso$"),                       None, None),
    ("ia_emp_motivo_alucinacao",             B, "3", ("sufixo", r"^falta_de_confiabilidade_das_saidas_alucinacao_dos_modelos$"),    "ia_emp_motivo_casos_uso", None),
    ("ia_emp_motivo_regulamentacao",         B, "3", ("sufixo", r"^incerteza_em_relacao_a_regulamentacao$"),                        "ia_emp_motivo_casos_uso", None),
    ("ia_emp_motivo_seguranca_privacidade",  B, "3", ("sufixo", r"^preocupacoes_com_seguranca_e_privacidade_de_dados$"),            "ia_emp_motivo_casos_uso", None),
    ("ia_emp_motivo_roi",                    B, "3", ("sufixo", r"^retorno_sobre_investimento_roi_nao_comprovado_de_ia_generativa$"), "ia_emp_motivo_casos_uso", None),
    ("ia_emp_motivo_dados_nao_prontos",      B, "3", ("sufixo", r"^dados_da_empresa_nao_estao_prontos_para_uso_de_ia_generativa$"), "ia_emp_motivo_casos_uso", None),
    ("ia_emp_motivo_expertise",              B, "3", ("sufixo", r"^falta_de_expertise_ou_falta_de_recursos$"),                      "ia_emp_motivo_casos_uso", None),
    ("ia_emp_motivo_alta_direcao",           B, "3", ("sufixo", r"^alta_direcao_da_empresa_nao_ve_valor_ou_nao_ve_como_prioridade$"), "ia_emp_motivo_casos_uso", None),
    ("ia_emp_motivo_propriedade_intelectual",B, "3", ("sufixo", r"^preocupacoes_com_propriedade_intelectual$"),                     "ia_emp_motivo_casos_uso", None),
    # só existe em 2025
    ("ia_emp_bons_resultados_llm",    S, "3", ("sufixo", r"^empresa_esta_conseguindo_ter_bons_resultados_com_llms$"), None, None),
    # IA - visão do indivíduo (seção 4, mesmos textos da seção 3)
    ("ia_ind_uso_independente",       B, "4", ("sufixo", r"usando_ai_generativa_de_forma_independente_e_descentralizada"), None, None),
    ("ia_ind_direcionamento_central", B, "4", ("sufixo", r"direcionamento_centralizado_do_uso_de_ai_generativa"), "ia_ind_uso_independente", None),
    ("ia_ind_copilots_dev",           B, "4", ("sufixo", r"desenvolvedores_utilizando_copilots"),                 "ia_ind_uso_independente", None),  # existe nos 3 anos
    ("ia_ind_produto_externo",        B, "4", ("sufixo", r"melhorar_produtos_externos"),                          "ia_ind_uso_independente", None),
    ("ia_ind_produto_interno",        B, "4", ("sufixo", r"melhorar_produtos_internos_para_os_colaboradores"),    "ia_ind_uso_independente", None),
    ("ia_ind_principal_frente",       B, "4", ("sufixo", r"como_principal_frente_do_negocio"),                    "ia_ind_uso_independente", None),
    ("ia_ind_nao_prioridade",         B, "4", ("sufixo", r"nao_e_prioridade"),                                    "ia_ind_uso_independente", None),
    ("ia_ind_nao_sei_opinar",         B, "4", ("sufixo", r"^nao_sei_opinar"),                                     "ia_ind_uso_independente", None),
    # IA - uso pessoal / produtividade
    ("ia_pessoal_nao_usa",            B, "4", ("sufixo", r"^nao_uso_solucoes_de_ai_generativa_com_foco_em_produtividade$"),          None, None),
    ("ia_pessoal_usa_gratis",         B, "4", ("sufixo", r"^uso_solucoes_gratuitas_de_ai_generativa_com_foco_em_produtividade$"),    "ia_pessoal_nao_usa", None),
    ("ia_pessoal_usa_pago_proprio",   B, "4", ("sufixo", r"^uso_e_pago_pelas_solucoes_de_ai_generativa_com_foco_em_produtividade$"), "ia_pessoal_nao_usa", None),
    ("ia_pessoal_usa_pago_empresa",   B, "4", ("sufixo", r"empresa_que_trabalho_paga_pelas_solucoes_de_ai_generativa_com_foco_em_produtividade$"), "ia_pessoal_nao_usa", None),
    ("ia_pessoal_usa_copilot",        B, "4", ("sufixo", r"^uso_solucoes_do_tipo_copilot$"),                                         "ia_pessoal_nao_usa", None),
]

# em 2024 o sub-item "Não utilizo nenhuma linguagem" veio 100% zero da fonte (bug do kaggle?),
# mas a guarda-chuva tem 155 respostas 'Não utilizo...' - então derivo de lá
TEC_USA_NENHUMA_GUARDA_CHUVA = {
    2024: ("4_d_linguagem_de_programacao_dia_a_dia", "Não utilizo"),
}

# texto livre da preferida -> categoria (lower + trim antes de comparar, tinha 'Sql', 'sql', 'Go'/'GO'/'Golang'...)
TEC_PREF_NORMALIZACAO = [
    ("Python",        ["python"]),
    ("SQL",           ["sql"]),
    ("R",             ["r"]),
    ("Go",            ["go", "golang"]),
    ("DAX/M",         ["dax/m", "m/dax", "m e dax", "dax"]),
    ("Java",          ["java"]),
    ("JavaScript",    ["javascript"]),
    ("C/C++/C#",      ["c/c++/c#"]),
    ("Scala",         ["scala"]),
    ("Rust",          ["rust"]),
    ("Julia",         ["julia"]),
    ("Não informado", ["não sei", "nao sei", "."]),
]
TEC_PREF_OUTRA = "Outra"

CAMPOS_DERIVADOS = ["tec_pref_normalizada", "ia_pessoal_nivel_uso"]

# colunas com _ na frente são auxiliares e ficam de fora
EXTENSAO_COLUMNS = [c[0] for c in CAMPOS_EXTENSAO if not c[0].startswith("_")] + CAMPOS_DERIVADOS

# partição por último
SILVER_COLUMNS = PERFIL_COLUMNS + EXTENSAO_COLUMNS + [PARTITION_COLUMN]

print(f"{len(CAMPOS_EXTENSAO)} campos da extensão + {len(CAMPOS_DERIVADOS)} derivado(s)")
print(f"Silver final: {len(SILVER_COLUMNS)} colunas")

58 campos da extensão + 2 derivado(s)
Silver final: 79 colunas


## 5. Leitura das tabelas catalogadas no Glue Data Catalog

As tabelas já estão disponíveis no Glue Catalog e podem ser consultadas pelo Athena. A leitura abaixo usa `create_dynamic_frame.from_catalog`. Como cada edição tem ~400 colunas, o schema completo não é impresso — apenas a quantidade de campos e a coluna de partição.

In [7]:
dyf_bronze = {}

for ano, table_name in TABLES_BRONZE.items():
    dyf_bronze[ano] = glueContext.create_dynamic_frame.from_catalog(
        database=DATABASE_NAME,
        table_name=table_name
    )
    campos = dyf_bronze[ano].schema().fields
    tipo_particao = [str(f.dataType) for f in campos if f.name == PARTITION_COLUMN]
    print(f"{table_name}: {len(campos)} campos | {PARTITION_COLUMN}: {tipo_particao}")

tb_pesquisa_2023_bronze: 400 campos | ano_pesquisa: ['StringType({})']
tb_pesquisa_2024_bronze: 404 campos | ano_pesquisa: ['StringType({})']
tb_pesquisa_2025_bronze: 389 campos | ano_pesquisa: ['StringType({})']


## 6. Conversão de DynamicFrame para DataFrame

A conversão para Spark DataFrame facilita transformações, seleção/renomeação de colunas e criação de colunas derivadas. As contagens abaixo servem de referência para a validação após o union.

In [7]:
df_bronze_raw = {}
qtd_bronze = {}

for ano, dyf in dyf_bronze.items():
    df_bronze_raw[ano] = dyf.toDF()
    qtd_bronze[ano] = df_bronze_raw[ano].count()
    print(f"Quantidade de registros em {TABLES_BRONZE[ano]}: {qtd_bronze[ano]} ({len(df_bronze_raw[ano].columns)} colunas)")

QTD_ESPERADA_TOTAL = sum(qtd_bronze.values())
print(f"\nTotal esperado após o union: {QTD_ESPERADA_TOTAL}")

Quantidade de registros em tb_pesquisa_2023_bronze: 5293 (400 colunas)
Quantidade de registros em tb_pesquisa_2024_bronze: 5217 (404 colunas)
Quantidade de registros em tb_pesquisa_2025_bronze: 3495 (389 colunas)

Total esperado após o union: 14005
/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


## 7. Resolução das colunas da extensão por ano

Para cada edição, resolve-se cada campo de `CAMPOS_EXTENSAO` para a coluna real da Bronze (ou `None`). A resolução é registrada em `resolucao_por_ano` para validação e para o arquivo de metadados. Uma regra que case com **mais de uma** coluna da mesma pergunta interrompe o processamento (ambiguidade), em vez de escolher silenciosamente.

In [7]:
def resolver_campo(df, indice, nome_silver, secao, regra, ancora, resolvidos):
    tipo_regra, valor = regra

    if tipo_regra == "nome":
        lookup = {c.lower(): c for c in df.columns}
        for nome in valor:
            if nome.lower() in lookup:
                return lookup[nome.lower()], f"nome exato: {nome}"
        return None, "nome exato: nenhum dos candidatos existe"

    # se tem âncora, só aceita coluna da mesma pergunta
    pergunta_ancora = None
    if ancora:
        col_ancora = resolvidos.get(ancora)
        if col_ancora is None:
            return None, f"âncora '{ancora}' não resolvida neste ano"
        m = CODIGO_PERGUNTA_RE.match(col_ancora)
        pergunta_ancora = codigo_pergunta_base(m.group(1))

    padrao = re.compile(valor)
    candidatos = [
        e for e in indice
        if e["secao"] == secao
        and padrao.search(e["sufixo"])
        and (pergunta_ancora is None or codigo_pergunta_base(e["codigo"]) == pergunta_ancora)
    ]
    if len(candidatos) == 1:
        return candidatos[0]["col"], f"sufixo /{valor}/ (seção {secao}" + (f", pergunta {pergunta_ancora}" if pergunta_ancora else "") + ")"
    if len(candidatos) > 1:
        raise ValueError(f"{nome_silver}: regra /{valor}/ ambígua na seção {secao}: {[c['col'] for c in candidatos]}")
    return None, f"sufixo /{valor}/ sem correspondência (seção {secao}" + (f", pergunta {pergunta_ancora}" if pergunta_ancora else "") + ")"


def resolver_extensao(df, ano: int):
    indice = indexar_sufixos(df)
    resolvidos = {}
    detalhes = {}
    for nome_silver, tipo, secao, regra, ancora, anos in CAMPOS_EXTENSAO:
        if anos is not None and ano not in anos:
            col, descricao = None, f"fora dos anos aplicáveis {anos}"
        else:
            col, descricao = resolver_campo(df, indice, nome_silver, secao, regra, ancora, resolvidos)
        resolvidos[nome_silver] = col
        detalhes[nome_silver] = (col, tipo, descricao)
    return detalhes


resolucao_por_ano = {}
for ano, df_raw in df_bronze_raw.items():
    resolucao_por_ano[ano] = resolver_extensao(df_raw, ano)
    encontrados = sum(1 for col, _, _ in resolucao_por_ano[ano].values() if col)
    print(f"{ano}: {encontrados}/{len(CAMPOS_EXTENSAO)} campos resolvidos")

print("\nDetalhe da resolução (campo | 2023 | 2024 | 2025):")
for nome_silver, _, _, _, _, _ in CAMPOS_EXTENSAO:
    cols = [resolucao_por_ano[a][nome_silver][0] or "-" for a in sorted(resolucao_por_ano)]
    print(f"  {nome_silver:<40} | " + " | ".join(cols))

2023: 47/58 campos resolvidos
2024: 47/58 campos resolvidos
2025: 41/58 campos resolvidos

Detalhe da resolução (campo | 2023 | 2024 | 2025):
  tec_usa_python                           | P4_d_3_Python | 4_d_3_Python | -
  tec_usa_sql                              | P4_d_1_SQL | 4_d_1_SQL | -
  tec_usa_r                                | P4_d_2_R | 4_d_2_R | -
  tec_usa_c_cpp_csharp                     | P4_d_4_C_C_C | 4_d_4_C_C_C | -
  tec_usa_julia                            | P4_d_7_Julia | 4_d_7_Julia | -
  tec_usa_vba                              | P4_d_9_Visual_Basic_VBA | 4_d_9_Visual_Basic_VBA | -
  tec_usa_scala                            | P4_d_10_Scala | 4_d_10_Scala | -
  tec_usa_rust                             | P4_d_12_Rust | 4_d_12_Rust | -
  tec_usa_nenhuma                          | P4_d_15_Não_utilizo_nenhuma_linguagem | 4_d_15_Não_utilizo_nenhuma_das_linguagens_listadas | -
  tec_usa_dotnet                           | P4_d_5_NET | 4_d_5_NET | -
  tec_usa_java          

## 8. Padronização do schema e harmonização de valores (por ano, antes do union)

Para cada edição são aplicados, nesta ordem:

1. **select/rename** conforme `COLUMN_MAPPING` (colunas ausentes na edição → `lit(None)`), preservando `ano_pesquisa`;
2. **colunas da extensão** conforme a resolução da seção 7 — sub-itens `"1"`/`"0"` → boolean, categóricos → string, ausentes/fora dos anos aplicáveis → `lit(None)`; em 2024, `tec_usa_nenhuma` é recalculado da guarda-chuva `4_d_linguagem_de_programacao_dia_a_dia` (`contains('Não utilizo')`);
3. **Regra 1** — `nivel_senioridade_agrupada`: `Especialista/Staff+` (só existe em 2025) agrupado em `Sênior`; demais valores mantidos;
4. **Regra 2** — correção dos typos em `faixa_salarial`;
5. **Regra 3** — `faixa_salarial_ordem` (Int) e `faixa_salarial_ponto_medio` (Double) a partir da faixa já corrigida; NULL ou valor não mapeado → NULL;
5b. **Regra 4** — `cargo_atual_agrupado`: harmoniza a fusão/separação de "Engenheiro de Dados" e "Arquiteto de Dados" entre edições da pesquisa (unidos em 2023, separados a partir de 2024) em um único rótulo `Engenheiro de Dados/Arquiteto de Dados`; demais cargos mantidos (incluindo NULL). Mesmo padrão de `nivel_senioridade_agrupada`;
6. **Derivação** — `tec_pref_normalizada` (2023/2024) a partir do texto livre de "linguagem preferida" via `TEC_PREF_NORMALIZACAO` (o que não casa vira `Outra`; 2025 fica NULL, pois lá a resposta é multi-select → `tec_pref_*`); `ia_pessoal_nivel_uso` a partir dos 5 booleanos de produtividade pessoal (precedência: empresa paga > paga do próprio bolso > gratuito/copilot > não usa).

Não é aplicado nenhum tratamento de NULL nas demais colunas: os nulos são estruturais (quem não trabalha com dados não responde senioridade/salário, por exemplo).

In [7]:
def resolve_column(df, nome_origem: str):
    # case-insensitive porque o glue catalog guarda os nomes em minúsculas e o parquet mantém o case original
    # (descobri isso na primeira execução: o from_catalog devolvia p1_b_genero e o mapeamento tinha P1_b_Genero)
    lookup = {c.lower(): c for c in df.columns}
    nome_real = lookup.get(nome_origem.lower())
    if nome_real is None:
        raise KeyError(f"Coluna de origem não encontrada: {nome_origem}")
    return F.col(f"`{nome_real}`")


def coluna_extensao(col_bronze, tipo: str, nome_silver: str):
    # '1'/'0' -> boolean; categórica -> string; ausente -> lit(None)
    if col_bronze is None:
        return F.lit(None).cast(BooleanType() if tipo == "boolean" else StringType()).alias(nome_silver)
    origem = F.col(f"`{col_bronze}`")
    if tipo == "boolean":
        return (
            F.when(origem == "1", F.lit(True))
             .when(origem == "0", F.lit(False))
             .otherwise(F.lit(None))
             .cast(BooleanType())
             .alias(nome_silver)
        )
    return origem.cast(StringType()).alias(nome_silver)


def padronizar_schema(df, ano: int):
    colunas = []

    for nome_silver, origem_por_ano in COLUMN_MAPPING.items():
        nome_origem = origem_por_ano.get(ano)
        if nome_origem is None:
            colunas.append(F.lit(None).cast(StringType()).alias(nome_silver))
        else:
            colunas.append(resolve_column(df, nome_origem).cast(StringType()).alias(nome_silver))

    for nome_silver, (col_bronze, tipo, _) in resolucao_por_ano[ano].items():
        expr = coluna_extensao(col_bronze, tipo, nome_silver)
        # caso especial 2024 (ver TEC_USA_NENHUMA_GUARDA_CHUVA)
        if nome_silver == "tec_usa_nenhuma" and ano in TEC_USA_NENHUMA_GUARDA_CHUVA:
            guarda_chuva, marcador = TEC_USA_NENHUMA_GUARDA_CHUVA[ano]
            expr = resolve_column(df, guarda_chuva).contains(marcador).cast(BooleanType()).alias(nome_silver)
        colunas.append(expr)

    colunas.append(resolve_column(df, PARTITION_COLUMN).cast(IntegerType()).alias(PARTITION_COLUMN))
    return df.select(*colunas)


def harmonizar_valores(df):

    # regra 1: Especialista/Staff+ só existe em 2025, agrupo em Sênior pra comparar com os outros anos
    df = df.withColumn(
        "nivel_senioridade_agrupada",
        F.when(F.col("nivel_senioridade") == "Especialista/Staff+", F.lit("Sênior"))
         .otherwise(F.col("nivel_senioridade"))
    )

    # regra 2: typos primeiro, senão a regra 3 não encontra a faixa
    expr_faixa = None
    for errado, correto in TYPOS_FAIXA_SALARIAL.items():
        cond = F.col("faixa_salarial") == errado
        expr_faixa = F.when(cond, F.lit(correto)) if expr_faixa is None else expr_faixa.when(cond, F.lit(correto))
    df = df.withColumn("faixa_salarial", expr_faixa.otherwise(F.col("faixa_salarial")))

    # regra 3
    expr_ordem = None
    expr_ponto_medio = None
    for rotulo, ordem, ponto_medio in FAIXA_SALARIAL_REF:
        cond = F.col("faixa_salarial") == rotulo
        expr_ordem = F.when(cond, F.lit(ordem)) if expr_ordem is None else expr_ordem.when(cond, F.lit(ordem))
        expr_ponto_medio = F.when(cond, F.lit(ponto_medio)) if expr_ponto_medio is None else expr_ponto_medio.when(cond, F.lit(ponto_medio))

    return (
        df
        .withColumn("faixa_salarial_ordem", expr_ordem.otherwise(F.lit(None)).cast(IntegerType()))
        .withColumn("faixa_salarial_ponto_medio", expr_ponto_medio.otherwise(F.lit(None)).cast(DoubleType()))
    )


def derivar_campos(df):
    # regra 4 (ver CARGOS_ENGENHEIRO_ARQUITETO_VARIANTES)
    df = df.withColumn(
        "cargo_atual_agrupado",
        F.when(F.col("cargo_atual").isin(CARGOS_ENGENHEIRO_ARQUITETO_VARIANTES), F.lit(CARGO_ENGENHEIRO_ARQUITETO))
         .otherwise(F.col("cargo_atual"))
    )

    # tec_pref_normalizada - em 2025 a coluna auxiliar é NULL, então fica NULL também (lá tem os booleanos tec_pref_*)
    texto = F.lower(F.trim(F.col("_tec_pref_texto_livre")))
    expr_pref = None
    for categoria, variantes in TEC_PREF_NORMALIZACAO:
        cond = texto.isin(variantes)
        expr_pref = F.when(cond, F.lit(categoria)) if expr_pref is None else expr_pref.when(cond, F.lit(categoria))
    df = df.withColumn(
        "tec_pref_normalizada",
        F.when(texto.isNull(), F.lit(None).cast(StringType())).otherwise(expr_pref.otherwise(F.lit(TEC_PREF_OUTRA)))
    )

    df = df.withColumn(
        "ia_pessoal_nivel_uso",
        F.when(F.col("ia_pessoal_usa_pago_empresa") == True, F.lit("Empresa paga"))
         .when(F.col("ia_pessoal_usa_pago_proprio") == True, F.lit("Paga do próprio bolso"))
         .when((F.col("ia_pessoal_usa_gratis") == True) | (F.col("ia_pessoal_usa_copilot") == True), F.lit("Usa gratuito/copilot"))
         .when(F.col("ia_pessoal_nao_usa") == True, F.lit("Não usa"))
         .otherwise(F.lit(None).cast(StringType()))
    )
    return df.select(*SILVER_COLUMNS)


df_silver_por_ano = {}
for ano, df_raw in df_bronze_raw.items():
    df_silver_por_ano[ano] = derivar_campos(harmonizar_valores(padronizar_schema(df_raw, ano)))
    print(f"{ano}: schema padronizado com {len(df_silver_por_ano[ano].columns)} colunas")

print("\nSchema tratado (2025):")
df_silver_por_ano[2025].printSchema()
df_silver_por_ano[2025].select(PERFIL_COLUMNS[:6] + ["tec_usa_python", "tec_pref_python", "tec_pref_sql", "ia_emp_uso_independente", "ia_pessoal_nivel_uso"]).show(5, truncate=False)

2023: schema padronizado com 79 colunas
2024: schema padronizado com 79 colunas
2025: schema padronizado com 79 colunas

Schema tratado (2025):
root
 |-- id_resposta: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- cor_raca_etnia: string (nullable = true)
 |-- uf_onde_mora: string (nullable = true)
 |-- regiao_onde_mora: string (nullable = true)
 |-- nivel_ensino: string (nullable = true)
 |-- area_formacao: string (nullable = true)
 |-- cargo_atual: string (nullable = true)
 |-- cargo_atual_agrupado: string (nullable = true)
 |-- nivel_senioridade: string (nullable = true)
 |-- nivel_senioridade_agrupada: string (nullable = true)
 |-- faixa_salarial: string (nullable = true)
 |-- faixa_salarial_ordem: integer (nullable = true)
 |-- faixa_salarial_ponto_medio: double (nullable = true)
 |-- setor: string (nullable = true)
 |-- numero_funcionarios: string (nullable = true)
 |-- tempo_experiencia_dados: string (nullable = true)
 |-- modelo_trabalho_atual: string (nulla

## 9. União das três edições da pesquisa

As edições são empilhadas com `unionByName` (alinhamento por nome de coluna, não por posição). `allowMissingColumns=True` cobre o caso de colunas ausentes em alguma edição.

In [7]:
df_state_of_data = None
for ano in sorted(df_silver_por_ano):
    df_ano = df_silver_por_ano[ano]
    df_state_of_data = df_ano if df_state_of_data is None else df_state_of_data.unionByName(df_ano, allowMissingColumns=True)

print(f"Quantidade de registros após union: {df_state_of_data.count()}")
print(f"Quantidade de colunas: {len(df_state_of_data.columns)}")

Quantidade de registros após union: 14005
Quantidade de colunas: 79


## 10. Validações da base unificada

1. Volume total e por ano contra as contagens da Bronze (nenhuma linha perdida/duplicada).
2. Regras 1–3 de harmonização do perfil.
3. Extensão — `tec_usa_*` / `tec_pref_*`: quantidade de `true` por ano, conferida contra valores já validados manualmente no Athena (`tec_usa_python` = 2825 / 2935; `tec_pref_python` 2025 = 1929; `tec_linguagem_mais_usada`: SQL = 1807 / 1760 e Python = 1358 / 1430). Se não bater, o sufixo resolveu a coluna errada.
4. Extensão — `ia_pessoal_nivel_uso`: distribuição por ano ao lado dos totais brutos dos 5 booleanos.
5. Extensão — lista dos campos que ficaram `lit(None)` em cada ano.
6. Tecnologia — `tec_usa_*` 100% NULL em 2025; `tec_usa_nenhuma` 2024 na ordem de grandeza dos 155 da guarda-chuva; `tec_pref_*` 100% NULL em 2023/2024; distribuição (top 15) de `tec_pref_normalizada` em 2023/2024 e NULL em 2025.

In [7]:
qtd_total = df_state_of_data.count()
print(f"Quantidade final de registros: {qtd_total} (esperado: {QTD_ESPERADA_TOTAL})")
assert qtd_total == QTD_ESPERADA_TOTAL, f"Volume total divergente: {qtd_total} != {QTD_ESPERADA_TOTAL}"

print("\nRegistros por ano_pesquisa (Silver x Bronze):")
qtd_silver_por_ano = {r[PARTITION_COLUMN]: r["qtd"] for r in df_state_of_data.groupBy(PARTITION_COLUMN).agg(F.count("*").alias("qtd")).collect()}
for ano in sorted(qtd_bronze):
    status = "OK" if qtd_silver_por_ano.get(ano) == qtd_bronze[ano] else "DIVERGENTE"
    print(f"   {ano}: silver={qtd_silver_por_ano.get(ano)} bronze={qtd_bronze[ano]} -> {status}")
    assert qtd_silver_por_ano.get(ano) == qtd_bronze[ano], f"Volume divergente em {ano}"

print("\nRegra 1 - nivel_senioridade x nivel_senioridade_agrupada:")
(
    df_state_of_data
    .groupBy("nivel_senioridade", "nivel_senioridade_agrupada")
    .agg(F.count("*").alias("qtd"))
    .orderBy(F.desc("qtd"))
    .show(truncate=False)
)

print("Regra 2 - typos remanescentes em faixa_salarial (esperado: 0):")
qtd_typos = df_state_of_data.filter(F.col("faixa_salarial").isin(list(TYPOS_FAIXA_SALARIAL.keys()))).count()
print(f"   {qtd_typos}")
assert qtd_typos == 0, "Ainda existem typos em faixa_salarial"

print("\nRegra 3 - faixa_salarial preenchida sem ordem/ponto médio (esperado: 0):")
qtd_sem_mapa = df_state_of_data.filter(F.col("faixa_salarial").isNotNull() & F.col("faixa_salarial_ordem").isNull()).count()
print(f"   {qtd_sem_mapa}")
assert qtd_sem_mapa == 0, "Existem faixas salariais sem mapeamento de ordem/ponto médio"

print("\nRegra 4 - cargo_atual x cargo_atual_agrupado por ano (só os rótulos afetados):")
(
    df_state_of_data
    .filter(F.col("cargo_atual").isin(CARGOS_ENGENHEIRO_ARQUITETO_VARIANTES) | (F.col("cargo_atual_agrupado") == CARGO_ENGENHEIRO_ARQUITETO))
    .groupBy("cargo_atual", "cargo_atual_agrupado").pivot(PARTITION_COLUMN).agg(F.count("*"))
    .orderBy("cargo_atual")
    .show(truncate=False)
)
qtd_variantes_restantes = df_state_of_data.filter(F.col("cargo_atual_agrupado").isin(CARGOS_ENGENHEIRO_ARQUITETO_VARIANTES)).count()
assert qtd_variantes_restantes == 0, "Ainda existem variantes de Engenheiro/Arquiteto de Dados em cargo_atual_agrupado"
qtd_demais_alterados = df_state_of_data.filter(~F.col("cargo_atual").isin(CARGOS_ENGENHEIRO_ARQUITETO_VARIANTES) & ~(F.col("cargo_atual").eqNullSafe(F.col("cargo_atual_agrupado")))).count()
assert qtd_demais_alterados == 0, "cargo_atual_agrupado alterou cargos fora das variantes mapeadas"
print("   Regra 4 OK: variantes restantes = 0; demais cargos inalterados (incluindo NULL)")

Quantidade final de registros: 14005 (esperado: 14005)

Registros por ano_pesquisa (Silver x Bronze):
   2023: silver=5293 bronze=5293 -> OK
   2024: silver=5217 bronze=5217 -> OK
   2025: silver=3495 bronze=3495 -> OK

Regra 1 - nivel_senioridade x nivel_senioridade_agrupada:
+-------------------+--------------------------+----+
|nivel_senioridade  |nivel_senioridade_agrupada|qtd |
+-------------------+--------------------------+----+
|Sênior             |Sênior                    |3850|
|NULL               |NULL                      |3829|
|Pleno              |Pleno                     |3545|
|Júnior             |Júnior                    |2432|
|Especialista/Staff+|Sênior                    |349 |
+-------------------+--------------------------+----+

Regra 2 - typos remanescentes em faixa_salarial (esperado: 0):
   0

Regra 3 - faixa_salarial preenchida sem ordem/ponto médio (esperado: 0):
   0

Regra 4 - cargo_atual x cargo_atual_agrupado por ano (só os rótulos afetados):
+-------

In [7]:
# validação 3 - tec_usa_*/tec_pref_* contra os números que conferi no athena
REFERENCIA_TRUE = {
    "tec_usa_python":  {2023: 2825, 2024: 2935},
    "tec_pref_python": {2025: 1929},
}
REFERENCIA_MAIS_USADA = {
    "SQL":    {2023: 1807, 2024: 1760},
    "Python": {2023: 1358, 2024: 1430},
}

campos_bool = [nome for nome, tipo, _, _, _, _ in CAMPOS_EXTENSAO if tipo == "boolean"]
aggs = [F.sum(F.col(c).cast("int")).alias(c) for c in campos_bool]
true_por_ano = {r[PARTITION_COLUMN]: r.asDict() for r in df_state_of_data.groupBy(PARTITION_COLUMN).agg(*aggs).collect()}
anos = sorted(true_por_ano)

print("Quantidade de TRUE por ano (campo | " + " | ".join(str(a) for a in anos) + "):")
for c in campos_bool:
    vals = [true_por_ano[a][c] for a in anos]
    print(f"  {c:<40} | " + " | ".join(f"{v if v is not None else '-':>5}" for v in vals))

for campo, esperado in REFERENCIA_TRUE.items():
    for ano, qtd in esperado.items():
        obtido = true_por_ano[ano][campo]
        assert obtido == qtd, f"{campo} em {ano}: obtido {obtido}, esperado {qtd} (sufixo resolveu a coluna errada?)"
print(f"\nReferências de TRUE conferidas: {REFERENCIA_TRUE}")

print("\ntec_linguagem_mais_usada por ano:")
mais_usada = (
    df_state_of_data.filter(F.col("tec_linguagem_mais_usada").isin(list(REFERENCIA_MAIS_USADA.keys())))
    .groupBy(PARTITION_COLUMN, "tec_linguagem_mais_usada").agg(F.count("*").alias("qtd")).collect()
)
obtido_mais_usada = {(r[PARTITION_COLUMN], r["tec_linguagem_mais_usada"]): r["qtd"] for r in mais_usada}
for linguagem, esperado in REFERENCIA_MAIS_USADA.items():
    for ano, qtd in esperado.items():
        obtido = obtido_mais_usada.get((ano, linguagem))
        status = "OK" if obtido == qtd else "DIVERGENTE"
        print(f"  {ano} {linguagem:<8} obtido={obtido} esperado={qtd} -> {status}")
        assert obtido == qtd, f"tec_linguagem_mais_usada {linguagem} em {ano}: obtido {obtido}, esperado {qtd}"
qtd_2025 = df_state_of_data.filter((F.col(PARTITION_COLUMN) == 2025) & F.col("tec_linguagem_mais_usada").isNotNull()).count()
print(f"  2025 (esperado sem valores): {qtd_2025} não nulos")

Quantidade de TRUE por ano (campo | 2023 | 2024 | 2025):
  tec_usa_python                           |  2825 |  2935 |     -
  tec_usa_sql                              |  3156 |  3146 |     -
  tec_usa_r                                |   408 |   352 |     -
  tec_usa_c_cpp_csharp                     |    70 |    46 |     -
  tec_usa_julia                            |     8 |     6 |     -
  tec_usa_vba                              |   285 |   215 |     -
  tec_usa_scala                            |   131 |   126 |     -
  tec_usa_rust                             |     7 |    12 |     -
  tec_usa_nenhuma                          |   305 |   165 |     -
  tec_usa_dotnet                           |    40 |    31 |     -
  tec_usa_java                             |   355 |   291 |     -
  tec_usa_sas_stata                        |   151 |   130 |     -
  tec_usa_matlab                           |    30 |    20 |     -
  tec_usa_php                              |    56 |    34 |     -
  tec

In [7]:
# validação 4 - ia_pessoal_nivel_uso
print("ia_pessoal_nivel_uso por ano_pesquisa:")
(
    df_state_of_data
    .groupBy(PARTITION_COLUMN, "ia_pessoal_nivel_uso")
    .agg(F.count("*").alias("qtd"))
    .orderBy(PARTITION_COLUMN, F.desc("qtd"))
    .show(30, truncate=False)
)

print("Totais brutos dos 5 booleanos de produtividade pessoal (TRUE por ano):")
campos_pessoal = [c for c in campos_bool if c.startswith("ia_pessoal_")]
for c in campos_pessoal:
    print(f"  {c:<32} | " + " | ".join(f"{true_por_ano[a][c] if true_por_ano[a][c] is not None else '-':>5}" for a in anos))

print("\nRespondentes com algum booleano pessoal preenchido (não nulo) por ano:")
(
    df_state_of_data
    .withColumn("respondeu_bloco", F.coalesce(*[F.col(c) for c in campos_pessoal]).isNotNull())
    .groupBy(PARTITION_COLUMN).agg(F.sum(F.col("respondeu_bloco").cast("int")).alias("qtd_respondeu"), F.sum(F.col("ia_pessoal_nivel_uso").isNotNull().cast("int")).alias("qtd_nivel_uso"))
    .orderBy(PARTITION_COLUMN)
    .show()
)

ia_pessoal_nivel_uso por ano_pesquisa:
+------------+---------------------+----+
|ano_pesquisa|ia_pessoal_nivel_uso |qtd |
+------------+---------------------+----+
|2023        |Usa gratuito/copilot |2560|
|2023        |NULL                 |1521|
|2023        |Não usa              |718 |
|2023        |Paga do próprio bolso|254 |
|2023        |Empresa paga         |240 |
|2024        |Usa gratuito/copilot |2174|
|2024        |NULL                 |1598|
|2024        |Empresa paga         |697 |
|2024        |Paga do próprio bolso|524 |
|2024        |Não usa              |224 |
|2025        |NULL                 |1389|
|2025        |Empresa paga         |892 |
|2025        |Usa gratuito/copilot |702 |
|2025        |Paga do próprio bolso|473 |
|2025        |Não usa              |39  |
+------------+---------------------+----+

Totais brutos dos 5 booleanos de produtividade pessoal (TRUE por ano):
  ia_pessoal_nao_usa               |   744 |   237 |    44
  ia_pessoal_usa_gratis         

In [7]:
# validação 5 - o que ficou lit(None) em cada ano tem que bater com o esperado
TEC_USA = {n for n, _, _, _, _, _ in CAMPOS_EXTENSAO if n.startswith("tec_usa_")}
TEC_PREF = {n for n, _, _, _, _, _ in CAMPOS_EXTENSAO if n.startswith("tec_pref_")}
ESPERADO_AUSENTE = {
    # ia_ind_copilots_dev existe nos 3 anos (achei que era só 2025, mas P4_l_3 e 4_l_3 existem)
    2023: TEC_PREF | {"ia_emp_bons_resultados_llm"},
    2024: TEC_PREF | {"ia_emp_bons_resultados_llm"},
    2025: TEC_USA | {"tec_linguagem_mais_usada", "_tec_pref_texto_livre"},
}

for ano in sorted(resolucao_por_ano):
    ausentes = {nome for nome, (col, _, _) in resolucao_por_ano[ano].items() if col is None}
    inesperados = ausentes - ESPERADO_AUSENTE[ano]
    faltando = ESPERADO_AUSENTE[ano] - ausentes
    print(f"{ano}: {len(ausentes)} campo(s) como lit(None): {sorted(ausentes)}")
    if inesperados:
        print(f"   ATENÇÃO - ausentes não esperados: {sorted(inesperados)}")
    if faltando:
        print(f"   ATENÇÃO - esperados ausentes mas resolvidos: {sorted(faltando)}")
    assert not inesperados and not faltando, f"Resolução de {ano} diverge do esperado"

print("\nResolução conforme o esperado nos 3 anos.")

2023: 11 campo(s) como lit(None): ['ia_emp_bons_resultados_llm', 'tec_pref_c_cpp_csharp', 'tec_pref_dax', 'tec_pref_julia', 'tec_pref_nenhuma', 'tec_pref_python', 'tec_pref_r', 'tec_pref_rust', 'tec_pref_scala', 'tec_pref_sql', 'tec_pref_vba']
2024: 11 campo(s) como lit(None): ['ia_emp_bons_resultados_llm', 'tec_pref_c_cpp_csharp', 'tec_pref_dax', 'tec_pref_julia', 'tec_pref_nenhuma', 'tec_pref_python', 'tec_pref_r', 'tec_pref_rust', 'tec_pref_scala', 'tec_pref_sql', 'tec_pref_vba']
2025: 17 campo(s) como lit(None): ['_tec_pref_texto_livre', 'tec_linguagem_mais_usada', 'tec_usa_c_cpp_csharp', 'tec_usa_dotnet', 'tec_usa_java', 'tec_usa_javascript', 'tec_usa_julia', 'tec_usa_matlab', 'tec_usa_nenhuma', 'tec_usa_php', 'tec_usa_python', 'tec_usa_r', 'tec_usa_rust', 'tec_usa_sas_stata', 'tec_usa_scala', 'tec_usa_sql', 'tec_usa_vba']

Resolução conforme o esperado nos 3 anos.


In [7]:
# validação 6 - correções do bloco de tecnologia
tec_usa_cols = sorted(TEC_USA)
tec_pref_cols = sorted(TEC_PREF)

print("tec_usa_* em 2025 - valores não nulos por coluna (esperado: 0 em todas):")
nao_nulos_2025 = df_state_of_data.filter(F.col(PARTITION_COLUMN) == 2025).agg(*[F.count(c).alias(c) for c in tec_usa_cols]).collect()[0].asDict()
for c, v in nao_nulos_2025.items():
    print(f"  {c:<26} {v}")
assert all(v == 0 for v in nao_nulos_2025.values()), "tec_usa_* deveria ser 100% NULL em 2025"

print("\ntec_pref_* em 2023/2024 - valores não nulos (esperado: 0):")
nao_nulos_pref = df_state_of_data.filter(F.col(PARTITION_COLUMN).isin(2023, 2024)).agg(*[F.count(c).alias(c) for c in tec_pref_cols]).collect()[0].asDict()
print(f"  {nao_nulos_pref}")
assert all(v == 0 for v in nao_nulos_pref.values()), "tec_pref_* deveria ser 100% NULL em 2023/2024"

print("\ntec_usa_nenhuma por ano (true / false / null) - 2024 esperado ~155 true:")
(
    df_state_of_data.groupBy(PARTITION_COLUMN)
    .agg(F.sum(F.col("tec_usa_nenhuma").cast("int")).alias("qtd_true"),
         F.sum((~F.col("tec_usa_nenhuma")).cast("int")).alias("qtd_false"),
         F.sum(F.col("tec_usa_nenhuma").isNull().cast("int")).alias("qtd_null"))
    .orderBy(PARTITION_COLUMN).show()
)

print("tec_pref_* em 2025 - quantidade de TRUE (tec_pref_python esperado 1929; 'Python, SQL' é a combinação mais comum):")
pref_2025 = df_state_of_data.filter(F.col(PARTITION_COLUMN) == 2025).agg(*[F.sum(F.col(c).cast("int")).alias(c) for c in tec_pref_cols]).collect()[0].asDict()
for c, v in pref_2025.items():
    print(f"  {c:<26} {v}")

print("\ntec_pref_normalizada por ano (top 15; 2025 esperado 100% NULL):")
for ano in sorted(TABLES_BRONZE):
    print(f"--- {ano}")
    (
        df_state_of_data.filter(F.col(PARTITION_COLUMN) == ano)
        .groupBy("tec_pref_normalizada").agg(F.count("*").alias("qtd"))
        .orderBy(F.desc("qtd")).show(15, truncate=False)
    )
qtd_pref_norm_2025 = df_state_of_data.filter((F.col(PARTITION_COLUMN) == 2025) & F.col("tec_pref_normalizada").isNotNull()).count()
assert qtd_pref_norm_2025 == 0, "tec_pref_normalizada deveria ser NULL em 2025"
print("Validação 6 concluída.")

tec_usa_* em 2025 - valores não nulos por coluna (esperado: 0 em todas):
  tec_usa_c_cpp_csharp       0
  tec_usa_dotnet             0
  tec_usa_java               0
  tec_usa_javascript         0
  tec_usa_julia              0
  tec_usa_matlab             0
  tec_usa_nenhuma            0
  tec_usa_php                0
  tec_usa_python             0
  tec_usa_r                  0
  tec_usa_rust               0
  tec_usa_sas_stata          0
  tec_usa_scala              0
  tec_usa_sql                0
  tec_usa_vba                0

tec_pref_* em 2023/2024 - valores não nulos (esperado: 0):
  {'tec_pref_c_cpp_csharp': 0, 'tec_pref_dax': 0, 'tec_pref_julia': 0, 'tec_pref_nenhuma': 0, 'tec_pref_python': 0, 'tec_pref_r': 0, 'tec_pref_rust': 0, 'tec_pref_scala': 0, 'tec_pref_sql': 0, 'tec_pref_vba': 0}

tec_usa_nenhuma por ano (true / false / null) - 2024 esperado ~155 true:
+------------+--------+---------+--------+
|ano_pesquisa|qtd_true|qtd_false|qtd_null|
+------------+--------+-------

## 11. Gravação da tabela unificada em Parquet no S3 e catalogação no Glue

A escrita abaixo cria/atualiza a tabela no Glue Catalog usando `saveAsTable` em modo `overwrite`, permitindo consulta posterior pelo Athena.

A saída é particionada por **`ano_pesquisa`**, mantendo a organização física por edição da pesquisa e reduzindo custo em consultas filtradas por ano. Como o schema ganhou colunas, a tabela do catálogo é removida antes para garantir que o novo schema seja registrado.

In [7]:
# drop antes porque o schema mudou (saveAsTable com overwrite não atualiza colunas novas no catálogo)
spark.sql(f"DROP TABLE IF EXISTS {OUTPUT_FULL_TABLE_NAME}")

(
    df_state_of_data
    .write
    .mode("overwrite")
    .format("parquet")
    .option("path", OUTPUT_PATH)
    .partitionBy(PARTITION_COLUMN)
    .saveAsTable(OUTPUT_FULL_TABLE_NAME)
)

print(f"Tabela criada/atualizada com sucesso: {OUTPUT_FULL_TABLE_NAME}")
print(f"Dados gravados em: {OUTPUT_PATH}")

Tabela criada/atualizada com sucesso: workspace.tb_state_of_data_silver
Dados gravados em: s3://614133392546-lab/data-output/silver/state-of-data/


## 12. Metadados de rastreabilidade (mapeamento Bronze → Silver efetivamente resolvido)

Grava em `data-output/silver/_metadata/` um JSON com, para cada coluna da Silver, a coluna de origem em cada ano, o tipo e a regra de resolução/derivação aplicada. É a contraparte do `data-output/bronze/_metadata/pesquisa-<ano>_colunas.json`.

In [7]:
def descrever_regra(regra):
    tipo_regra, valor = regra
    return f"nome exato: {valor}" if tipo_regra == "nome" else f"sufixo descritivo ~ /{valor}/"

colunas_meta = {}
for nome_silver, origem_por_ano in COLUMN_MAPPING.items():
    colunas_meta[nome_silver] = {
        "bloco": "perfil", "tipo": "string",
        "origem": {str(a): origem_por_ano.get(a) for a in sorted(TABLES_BRONZE)},
        "regra": "mapeamento explícito por nome (COLUMN_MAPPING)",
    }
colunas_meta["nivel_senioridade_agrupada"] = {"bloco": "perfil", "tipo": "string", "origem": "derivada de nivel_senioridade", "regra": "'Especialista/Staff+' -> 'Sênior'; demais valores mantidos"}
colunas_meta["cargo_atual_agrupado"] = {
    "bloco": "perfil", "tipo": "string", "origem": "derivada de cargo_atual",
    "regra": {"variantes": CARGOS_ENGENHEIRO_ARQUITETO_VARIANTES, "rotulo_unico": CARGO_ENGENHEIRO_ARQUITETO, "demais": "mantidos (incluindo NULL)"},
    "nota": "cargo_atual_agrupado harmoniza a fusão/separação de 'Engenheiro de Dados' e 'Arquiteto de Dados' entre edições da pesquisa (unidos em 2023, separados a partir de 2024)",
}
colunas_meta["faixa_salarial"]["regra"] += f"; typos corrigidos: {TYPOS_FAIXA_SALARIAL}"
colunas_meta["faixa_salarial_ordem"] = {"bloco": "perfil", "tipo": "int", "origem": "derivada de faixa_salarial", "regra": {r: o for r, o, _ in FAIXA_SALARIAL_REF}}
colunas_meta["faixa_salarial_ponto_medio"] = {"bloco": "perfil", "tipo": "double", "origem": "derivada de faixa_salarial", "regra": {r: p for r, _, p in FAIXA_SALARIAL_REF}}

for nome_silver, tipo, secao, regra, ancora, anos in CAMPOS_EXTENSAO:
    if nome_silver.startswith("_"):
        continue  # coluna auxiliar, não vai para a Silver
    bloco = nome_silver.split("_")[0] if not nome_silver.startswith("ia_") else "_".join(nome_silver.split("_")[:2])
    colunas_meta[nome_silver] = {
        "bloco": bloco, "tipo": tipo,
        "origem": {str(a): resolucao_por_ano[a][nome_silver][0] for a in sorted(TABLES_BRONZE)},
        "anos_aplicaveis": list(anos) if anos else sorted(TABLES_BRONZE),
        "regra": descrever_regra(regra) + f" | seção {secao}" + (f" | mesma pergunta de '{ancora}'" if ancora else ""),
        "conversao": "'1' -> true, '0' -> false, null -> null" if tipo == "boolean" else "cast string",
    }
for ano, (guarda_chuva, marcador) in TEC_USA_NENHUMA_GUARDA_CHUVA.items():
    colunas_meta["tec_usa_nenhuma"]["origem"][str(ano)] = guarda_chuva
    colunas_meta["tec_usa_nenhuma"]["regra"] += f" | {ano}: derivado da guarda-chuva contains('{marcador}') (sub-item não codificado na fonte)"
colunas_meta["tec_pref_normalizada"] = {
    "bloco": "tec", "tipo": "string", "anos_aplicaveis": [2023, 2024],
    "origem": {str(a): resolucao_por_ano[a]["_tec_pref_texto_livre"][0] for a in sorted(TABLES_BRONZE)},
    "regra": {"normalizacao": "lower(trim(texto)) in variantes -> categoria; sem correspondência -> 'Outra'; null -> null; 2025 -> null (multi-select, ver tec_pref_*)",
              "categorias": {cat: vs for cat, vs in TEC_PREF_NORMALIZACAO}},
}
colunas_meta["ia_pessoal_nivel_uso"] = {
    "bloco": "ia_pessoal", "tipo": "string", "origem": "derivada de ia_pessoal_usa_pago_empresa / usa_pago_proprio / usa_gratis / usa_copilot / nao_usa",
    "regra": ["usa_pago_empresa -> 'Empresa paga'", "usa_pago_proprio -> 'Paga do próprio bolso'", "usa_gratis OR usa_copilot -> 'Usa gratuito/copilot'", "nao_usa -> 'Não usa'", "senão -> null"],
}
colunas_meta[PARTITION_COLUMN] = {"bloco": "particao", "tipo": "int", "origem": {str(a): PARTITION_COLUMN for a in sorted(TABLES_BRONZE)}, "regra": "partição herdada da Bronze"}

NOTA_METODOLOGICA = (
    "A partir de 2025, a pesquisa não repete a pergunta 'linguagens usadas no dia a dia' (substituída por "
    "'linguagem preferida', de natureza diferente) - comparação de adoção de tecnologia 2023->2025 fica limitada a "
    "2023-2024; 2025 é reportado separadamente como 'linguagem preferida' (tec_pref_*)."
)

metadata = {
    "tabela": OUTPUT_FULL_TABLE_NAME,
    "nota_metodologica": NOTA_METODOLOGICA,
    "location": OUTPUT_PATH,
    "particao": PARTITION_COLUMN,
    "origens": {str(a): f"{DATABASE_NAME}.{t}" for a, t in TABLES_BRONZE.items()},
    "total_colunas": len(SILVER_COLUMNS),
    "regex_codigo_pergunta": CODIGO_PERGUNTA_RE.pattern,
    "colunas": {c: colunas_meta[c] for c in SILVER_COLUMNS},
}
boto3.client("s3").put_object(
    Bucket=BUCKET_NAME, Key=METADATA_KEY,
    Body=json.dumps(metadata, ensure_ascii=False, indent=2).encode("utf-8"),
    ContentType="application/json",
)
print(f"Metadados gravados em: s3://{BUCKET_NAME}/{METADATA_KEY} ({len(metadata['colunas'])} colunas)")

Metadados gravados em: s3://614133392546-lab/data-output/silver/_metadata/tb_state_of_data_silver_colunas.json (79 colunas)


## 13. Consulta de validação da tabela catalogada

Após a gravação, a tabela pode ser consultada via Spark SQL no próprio notebook ou pelo Athena. A taxa de preenchimento de `nivel_senioridade` e `faixa_salarial` por ano deve bater com o perfil já levantado na Bronze (NULLs estruturais: ~27% em senioridade e ~7–10% em salário).

In [7]:
spark.sql(f"SHOW TABLES IN {DATABASE_NAME}").show(truncate=False)

spark.sql(f"""
    SELECT
        ano_pesquisa,
        COUNT(*)                  AS qtd_respostas,
        COUNT(nivel_senioridade)  AS qtd_nivel_senioridade,
        COUNT(faixa_salarial)     AS qtd_faixa_salarial,
        ROUND(100.0 * COUNT(nivel_senioridade) / COUNT(*), 1) AS pct_nivel_senioridade,
        ROUND(100.0 * COUNT(faixa_salarial) / COUNT(*), 1)    AS pct_faixa_salarial,
        SUM(CASE WHEN tec_usa_python THEN 1 ELSE 0 END)       AS qtd_usa_python,
        SUM(CASE WHEN tec_usa_sql THEN 1 ELSE 0 END)          AS qtd_usa_sql,
        COUNT(ia_pessoal_nivel_uso)                           AS qtd_nivel_uso_ia
    FROM {OUTPUT_FULL_TABLE_NAME}
    GROUP BY ano_pesquisa
    ORDER BY ano_pesquisa
""").show(truncate=False)

+---------+---------------------------------+-----------+
|namespace|tableName                        |isTemporary|
+---------+---------------------------------+-----------+
|workspace|tb_gold_adocao_ia_pessoal        |false      |
|workspace|tb_gold_adocao_ia_uso            |false      |
|workspace|tb_gold_adocao_tecnologia        |false      |
|workspace|tb_gold_barreiras_ia             |false      |
|workspace|tb_gold_diversidade              |false      |
|workspace|tb_gold_perfil_mercado           |false      |
|workspace|tb_gold_regiao_senioridade_modelo|false      |
|workspace|tb_gold_salario_por_perfil       |false      |
|workspace|tb_pesquisa_2023_bronze          |false      |
|workspace|tb_pesquisa_2024_bronze          |false      |
|workspace|tb_pesquisa_2025_bronze          |false      |
|workspace|tb_state_of_data_silver          |false      |
+---------+---------------------------------+-----------+

+------------+-------------+---------------------+------------------+--

## 14. Encerramento opcional do job

Em um Glue Job produtivo, use `job.commit()` ao final. Em sessão interativa, esta célula pode ser mantida comentada se preferir.

In [7]:
# job.commit()